# **Sistem Pendeteksi Objek**

## Clone Repo Github untuk kebutuhan lib YoloV7

In [ ]:
# Clone repositori YOLOv7
!git clone https://github.com/WongKinYiu/yolov7.git

# Masuk ke direktori yolov7
%cd yolov7

# Unduh pre-trained model YOLOv7
!wget https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt

# Catatan: Baris '!pip install -qr requirements.txt' KITA HAPUS
# karena Google Colab sudah memiliki environment yang cukup untuk inferensi.
print("Persiapan file dan model selesai!")

Cloning into 'yolov7'...
remote: Enumerating objects: 1197, done.
remote: Total 1197 (delta 0), reused 0 (delta 0), pack-reused 1197 (from 1)
Receiving objects: 100% (1197/1197), 74.29 MiB | 28.59 MiB/s, done.
Resolving deltas: 100% (511/511), done.
/content/yolov7
--2026-02-26 09:07:58--  https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt
Resolving github.com (github.com)... 140.82.121.3
Connecting to github.com (github.com)|140.82.121.3|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/511187726/b0243edf-9fb0-4337-95e1-42555f1b37cf?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-02-26T10%3A04%3A30Z&rscd=attachment%3B+filename%3Dyolov7.pt&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-02-26T09%3A03%3A39Z&ske=2026-02-26T10%3A04%3A30Z&sks=b&skv=2018-11-09&sig=lcIrcajwTkLPfUGn81fiDanMOq6zrLqj

## **Memuat Model YOLOv7 & Menyiapkan Warna Label**



In [ ]:
import torch
import cv2
import numpy as np
import random
import torch.nn as nn
from models.experimental import attempt_load
from utils.general import non_max_suppression, scale_coords
from utils.datasets import letterbox
from utils.plots import plot_one_box

# Compatibility for newer PyTorch versions
from models.yolo import Model, IDetect, Detect
from models.common import Conv, MP, SPPCSPC, Concat, RepConv

if hasattr(torch.serialization, 'add_safe_globals'):
    torch.serialization.add_safe_globals([
        Model, IDetect, Detect, Conv, MP, SPPCSPC, Concat, RepConv,
        nn.modules.container.Sequential,
        nn.modules.conv.Conv2d,
        nn.modules.upsampling.Upsample,
        nn.modules.activation.SiLU,
        nn.modules.batchnorm.BatchNorm2d,
        nn.modules.container.ModuleList,
        nn.modules.pooling.MaxPool2d
    ])

# Gunakan GPU jika tersedia
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Menggunakan perangkat: {device}")

# Muat model YOLOv7
model = attempt_load('yolov7.pt', map_location=device)
model.eval()

# Ambil nama label
names = model.module.names if hasattr(model, 'module') else model.names
colors = [[random.randint(0, 255) for _ in range(3)] for _ in names]

def detect_frame(img_bgr):
    # Resize & Normalisasi
    img_resized = letterbox(img_bgr, 640, stride=32)[0]
    # .copy() handles negative strides from slicing/transposing
    img_rgb = img_resized[:, :, ::-1].transpose(2, 0, 1).copy()
    img_tensor = torch.from_numpy(img_rgb).to(device)
    img_tensor = img_tensor.float() / 255.0

    if img_tensor.ndimension() == 3:
        img_tensor = img_tensor.unsqueeze(0)

    # Inferensi
    with torch.no_grad():
        pred = model(img_tensor, augment=False)[0]

    # NMS
    pred = non_max_suppression(pred, 0.25, 0.45, classes=None, agnostic=False)

    for i, det in enumerate(pred):
        if len(det):
            det[:, :4] = scale_coords(img_tensor.shape[2:], det[:, :4], img_bgr.shape).round()
            for *xyxy, conf, cls in reversed(det):
                label_text = f'{names[int(cls)]} {conf:.2f}'
                plot_one_box(xyxy, img_bgr, label=label_text, color=colors[int(cls)], line_thickness=2)

    return img_bgr

Menggunakan perangkat: cuda
Fusing layers... 
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block
RepConv.fuse_repvgg_block


## **Akses Webcam Real-time dengan JavaScript**

In [ ]:
import base64
import html
import io
import time
from IPython.display import display, Javascript
from google.colab.output import eval_js
from PIL import Image

# Konversi JavaScript Image ke OpenCV
def js_to_image(js_reply):
  image_bytes = base64.b64decode(js_reply.split(',')[1])
  jpg_as_np = np.frombuffer(image_bytes, dtype=np.uint8)
  img = cv2.imdecode(jpg_as_np, flags=1)
  return img

# Konversi OpenCV ke base64 (untuk ditampilkan di HTML)
def bbox_to_bytes(bbox_array):
  _, bbox_encoded = cv2.imencode('.jpg', bbox_array)
  bbox_bytes = base64.b64encode(bbox_encoded).decode('utf-8')
  return bbox_bytes

# Fungsi JavaScript untuk mengaktifkan video stream
def video_stream():
  js = Javascript('''
    var video;
    var div = null;
    var stream;
    var captureCanvas;
    var imgElement;
    var labelElement;

    var pendingResolve = null;
    var shutdown = false;

    function removeDom() {
       shutdown = true;
       if (video) {
         video.srcObject.getTracks().forEach(function(track) { track.stop(); });
         video.remove();
       }
       if (div) { div.remove(); }
       if (stream) { stream.getVideoTracks()[0].stop(); }
    }

    function onAnimationFrame() {
      if (!shutdown) {
        window.requestAnimationFrame(onAnimationFrame);
      }
      if (pendingResolve) {
        var result = "";
        if (!shutdown) {
          captureCanvas.getContext('2d').drawImage(video, 0, 0, 640, 480);
          result = captureCanvas.toDataURL('image/jpeg', 0.8)
        }
        var lp = pendingResolve;
        pendingResolve = null;
        lp(result);
      }
    }

    async function createDom() {
      if (div !== null) { return stream; }
      div = document.createElement('div');
      div.style.border = '2px solid black';
      div.style.padding = '3px';
      div.style.width = '100%';
      div.style.maxWidth = '600px';
      document.body.appendChild(div);

      const modelOut = document.createElement('div');
      modelOut.innerHTML = "<span>Status: Menyalakan Kamera...</span>";
      labelElement = modelOut;
      div.appendChild(modelOut);

      video = document.createElement('video');
      video.style.display = 'block';
      video.width = div.clientWidth - 6;
      video.setAttribute('playsinline', '');
      video.onclick = () => { shutdown = true; };
      stream = await navigator.mediaDevices.getUserMedia(
          {video: { facingMode: "environment"}});
      div.appendChild(video);
      imgElement = document.createElement('img');
      imgElement.style.position = 'absolute';
      imgElement.style.zIndex = 1;
      imgElement.onclick = () => { shutdown = true; };
      div.appendChild(imgElement);

      const instruction = document.createElement('div');
      instruction.innerHTML = '<span style="color: red; font-weight: bold;">Klik pada area video untuk BERHENTI</span>';
      div.appendChild(instruction);

      video.srcObject = stream;
      await video.play();
      captureCanvas = document.createElement('canvas');
      captureCanvas.width = 640;
      captureCanvas.height = 480;
      window.requestAnimationFrame(onAnimationFrame);

      return stream;
    }
    async function stream_frame(label, imgData) {
      if (shutdown) {
        removeDom();
        shutdown = false;
        return '';
      }
      var preCreate = Date.now();
      stream = await createDom();

      var preShow = Date.now();
      if (label != "") {
        labelElement.innerHTML = label;
      }

      if (imgData != "") {
        var videoRect = video.getClientRects()[0];
        imgElement.style.top = videoRect.top + "px";
        imgElement.style.left = videoRect.left + "px";
        imgElement.style.width = videoRect.width + "px";
        imgElement.style.height = videoRect.height + "px";
        imgElement.src = imgData;
      }

      var preCapture = Date.now();
      var result = await new Promise(function(resolve, reject) {
        pendingResolve = resolve;
      });
      shutdown = false;
      return {'create': preShow - preCreate,
              'show': preCapture - preShow,
              'capture': Date.now() - preCapture,
              'img': result};
    }
    ''')
  display(js)

def video_frame(label, bbox):
  data = eval_js('stream_frame("{}", "{}")'.format(label, bbox))
  return data

# --- EKSEKUSI PROGRAM UTAMA ---
video_stream()
label_html = 'Sistem Deteksi YOLOv7 Aktif...'
bbox = ''
count = 0

while True:
    try:
        js_reply = video_frame(label_html, bbox)
        if not js_reply:
            break

        # Ambil frame dari webcam
        img = js_to_image(js_reply["img"])

        # Lakukan deteksi objek
        result_img = detect_frame(img)

        # Ubah gambar hasil deteksi ke base64 agar bisa dirender di HTML
        bbox = 'data:image/jpeg;base64,' + bbox_to_bytes(result_img)

    except Exception as e:
        print(f"Berhenti: {e}")
        break

<IPython.core.display.Javascript object>

Berhenti: NotAllowedError: Permission denied


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("andrewmvd/face-mask-detection")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'face-mask-detection' dataset.
Path to dataset files: /kaggle/input/face-mask-detection
